In [2]:
println!("Hello, world from Rust in Jupyter!");

Hello, world from Rust in Jupyter!


In [8]:
:dep polars = { version = "0.46", features = ["csv", "lazy", "describe", "dtype-date"] }

use polars::prelude::*;

let df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("fruits.csv".into()))?
    .finish()?;

println!("{}", df);

shape: (16, 4)
┌────────┬───────┬──────────┬────────────┐
│ fruit  ┆ price ┆ quantity ┆ date       │
│ ---    ┆ ---   ┆ ---      ┆ ---        │
│ str    ┆ f64   ┆ i64      ┆ str        │
╞════════╪═══════╪══════════╪════════════╡
│ Apple  ┆ 1.2   ┆ 50       ┆ 2025-01-01 │
│ Banana ┆ 0.5   ┆ 120      ┆ 2025-01-01 │
│ Orange ┆ 0.8   ┆ 75       ┆ 2025-01-01 │
│ Mango  ┆ 2.5   ┆ 30       ┆ 2025-01-01 │
│ Apple  ┆ 1.25  ┆ 45       ┆ 2025-02-01 │
│ …      ┆ …     ┆ …        ┆ …          │
│ Mango  ┆ 2.6   ┆ 25       ┆ 2025-03-01 │
│ Apple  ┆ 1.15  ┆ 55       ┆ 2025-04-01 │
│ Banana ┆ 0.52  ┆ 115      ┆ 2025-04-01 │
│ Orange ┆ 0.78  ┆ 85       ┆ 2025-04-01 │
│ Mango  ┆ 2.45  ┆ 32       ┆ 2025-04-01 │
└────────┴───────┴──────────┴────────────┘


In [10]:
// .head() — first 5 rows
println!("=== Head ===");
println!("{}", df.head(Some(5)));

// .describe() — summary statistics
println!("\n=== Describe ===");
println!("{:?}", df.describe()?);

Error: no method named `describe` found for struct `polars::prelude::DataFrame` in the current scope

In [11]:
// Average price and total quantity per fruit
let grouped = df.clone().lazy()
    .group_by([col("fruit")])
    .agg([
        col("price").mean().alias("avg_price"),
        col("quantity").sum().alias("total_quantity"),
    ])
    .collect()?;

println!("=== Group By Fruit ===");
println!("{}", grouped);

=== Group By Fruit ===
shape: (4, 3)
┌────────┬───────────┬────────────────┐
│ fruit  ┆ avg_price ┆ total_quantity │
│ ---    ┆ ---       ┆ ---            │
│ str    ┆ f64       ┆ i64            │
╞════════╪═══════════╪════════════════╡
│ Apple  ┆ 1.225     ┆ 190            │
│ Mango  ┆ 2.4875    ┆ 122            │
│ Banana ┆ 0.5125    ┆ 475            │
│ Orange ┆ 0.8325    ┆ 310            │
└────────┴───────────┴────────────────┘


In [12]:
// Filter: fruits with price > 1.00, sorted by price descending
let filtered = df.clone().lazy()
    .filter(col("price").gt(lit(1.0)))
    .sort(["price"], SortMultipleOptions::default().with_order_descending(true))
    .collect()?;

println!("=== Fruits with price > 1.00 (sorted desc) ===");
println!("{}", filtered);

=== Fruits with price > 1.00 (sorted desc) ===
shape: (8, 4)
┌───────┬───────┬──────────┬────────────┐
│ fruit ┆ price ┆ quantity ┆ date       │
│ ---   ┆ ---   ┆ ---      ┆ ---        │
│ str   ┆ f64   ┆ i64      ┆ str        │
╞═══════╪═══════╪══════════╪════════════╡
│ Mango ┆ 2.6   ┆ 25       ┆ 2025-03-01 │
│ Mango ┆ 2.5   ┆ 30       ┆ 2025-01-01 │
│ Mango ┆ 2.45  ┆ 32       ┆ 2025-04-01 │
│ Mango ┆ 2.4   ┆ 35       ┆ 2025-02-01 │
│ Apple ┆ 1.3   ┆ 40       ┆ 2025-03-01 │
│ Apple ┆ 1.25  ┆ 45       ┆ 2025-02-01 │
│ Apple ┆ 1.2   ┆ 50       ┆ 2025-01-01 │
│ Apple ┆ 1.15  ┆ 55       ┆ 2025-04-01 │
└───────┴───────┴──────────┴────────────┘


In [13]:
let multi_sorted = df.clone().lazy()
    .sort(
        ["fruit", "price"],
        SortMultipleOptions::default()
            .with_order_descending_multi([false, true]),
    )
    .collect()?;

println!("=== Sorted by fruit ASC, price DESC ===");
println!("{}", multi_sorted);

=== Sorted by fruit ASC, price DESC ===
shape: (16, 4)
┌────────┬───────┬──────────┬────────────┐
│ fruit  ┆ price ┆ quantity ┆ date       │
│ ---    ┆ ---   ┆ ---      ┆ ---        │
│ str    ┆ f64   ┆ i64      ┆ str        │
╞════════╪═══════╪══════════╪════════════╡
│ Apple  ┆ 1.3   ┆ 40       ┆ 2025-03-01 │
│ Apple  ┆ 1.25  ┆ 45       ┆ 2025-02-01 │
│ Apple  ┆ 1.2   ┆ 50       ┆ 2025-01-01 │
│ Apple  ┆ 1.15  ┆ 55       ┆ 2025-04-01 │
│ Banana ┆ 0.55  ┆ 110      ┆ 2025-02-01 │
│ …      ┆ …     ┆ …        ┆ …          │
│ Mango  ┆ 2.4   ┆ 35       ┆ 2025-02-01 │
│ Orange ┆ 0.9   ┆ 70       ┆ 2025-03-01 │
│ Orange ┆ 0.85  ┆ 80       ┆ 2025-02-01 │
│ Orange ┆ 0.8   ┆ 75       ┆ 2025-01-01 │
│ Orange ┆ 0.78  ┆ 85       ┆ 2025-04-01 │
└────────┴───────┴──────────┴────────────┘
